# 2025 Midterm Halalan Sentiment Analysis

In [1]:
!python --version
!nvcc --version

Python 3.12.7
/bin/bash: line 1: nvcc: command not found


# Install Dependencies

In [2]:
import pydantic
pydantic.__version__

'2.11.10'

In [3]:
# Import necessary libraries

from time import sleep
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os

import sys

from IPython.display import Markdown as md

# Model

import instructor
from groq import Groq

In [4]:
# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive, userdata
    GROQ_TOKEN = userdata.get('GROQ_TOKEN')
else:
    from dotenv import load_dotenv
    load_dotenv(dotenv_path='.env')
    GROQ_TOKEN = os.getenv("GROQ_TOKEN")

print(GROQ_TOKEN[-3:])

vp3


# Setup Instructor and Model

In [5]:
# Initialize Groq client

client = instructor.from_groq(
    Groq(api_key=GROQ_TOKEN),
    mode=instructor.Mode.JSON
)


# Indicate Groq Model, llama-4 maverick
MODEL_ID = 'meta-llama/llama-4-maverick-17b-128e-instruct'


In [6]:
system_prompt = """
You are a model that does sentiment analysis fluent in both English and various Philippine languages, including but not limited to Filipino, Cebuano, Ilocano, Kapampangan, Bicolano, a mix of Filipino and English, among others.

With an extensive background in the local culture and a deep understanding of the current political status of the Philippines with respect to the midterm elections, regards to the Halalan 2025 midterm election situated in the Republic of the Philippines, and their sentiments based on the given context.

Your primary task is to accurately extract structured features from form responses, including tone, perceived judgement with regards to a particular candidate while understanding that typographical errors and the similarity of names but not last names are bound to happen but must always be cautiously checked if it qualifies the mentioned names above; you must recognize that the user can mention more than one name within its content, which must also be included within your output; to add, determine the best suited word in the content that describes the associated candidate/partylist mentioned, which must be at most 1 word always and must be included inside the content, do not deviate from other descriptors. Amongst all of this, polarity must also be analyzed.

Your responses should be concise, contextually accurate, and reflective of the diverse linguistic nuances in Philippine social media.
"""
md(system_prompt)


You are a model that does sentiment analysis fluent in both English and various Philippine languages, including but not limited to Filipino, Cebuano, Ilocano, Kapampangan, Bicolano, a mix of Filipino and English, among others.

With an extensive background in the local culture and a deep understanding of the current political status of the Philippines with respect to the midterm elections, regards to the Halalan 2025 midterm election situated in the Republic of the Philippines, and their sentiments based on the given context.

Your primary task is to accurately extract structured features from form responses, including tone, perceived judgement with regards to a particular candidate while understanding that typographical errors and the similarity of names but not last names are bound to happen but must always be cautiously checked if it qualifies the mentioned names above; you must recognize that the user can mention more than one name within its content, which must also be included within your output; to add, determine the best suited word in the content that describes the associated candidate/partylist mentioned, which must be at most 1 word always and must be included inside the content, do not deviate from other descriptors. Amongst all of this, polarity must also be analyzed.

Your responses should be concise, contextually accurate, and reflective of the diverse linguistic nuances in Philippine social media.


In [7]:
from typing import Literal

Candidates = ['Benhur Abalos', 'Jerome Adonis', 'Wilson Amad', 'Jocelyn Andamo', 'Bam Aquino', 'Ronnel Arambulo', 'Ernesto Arellano', 'Roberto Ballon', 'Abigail Binay', 'Jimmy Bondoc', 'Bong Revilla', 'Bonifacio Bosita', 'Arlene Brosas', 'Roy Cabonegro', 'Allen Capuyan', 'Teodoro Casiño', 'France Castro', 'Pia Cayetano', "David d'Angelo", 'Angelo de Alban', 'Leody de Guzman', 'Ronald dela Rosa', 'Mimi Doringo', 'Arnel Escobal', 'Luke Espiritu', 'Mody Floranda', 'Marc Gamboa', 'Bong Go', 'Norberto Gonzales', 'Jesus Hinlo Jr.', 'Gregorio Honasan', 'Relly Jose Jr.', 'Panfilo Lacson', 'Raul Lambino', 'Lito Lapid', 'Wilbert T. Lee', 'Amirah Lidasan', 'Rodante Marcoleta', 'Imee Marcos', 'Norman Marquez', 'Eric Martinez', 'Richard Mata', 'Sonny Matula', 'Liza Maza', 'Heidi Mendoza', 'Jose Montemayor Jr.', 'Subair Mustapha', 'Jose Olivar', 'Willie Ong', 'Manny Pacquiao', 'Kiko Pangilinan', 'Ariel Querubin', 'Apollo Quiboloy', 'Danilo Ramos', 'Willie Revillame', 'Vic Rodriguez', 'Nur-Ana Sahidulla', 'Phillip Salvador', 'Tito Sotto', 'Michael Tapado', 'Francis Tolentino', 'Ben Tulfo', 'Erwin Tulfo', 'Mar Valbuena', 'Leandro Verceles Jr.', 'Camille Villar']
Partylists = ['Makabayan', '4Ps', 'PPP', 'FPJ Panday Bayanihan', 'Kabataan', 'Duterte Youth', 'ML', 'PBBM', 'P3PWD', 'Murang Kuryente', 'Bicol Saro', 'Ipatupad', 'PATROL', 'Juan PINOY', 'ARTE', 'WIFI', 'MAAGAP', 'United Senior Citizens', 'Epanaw Sambayanan', 'Ako Padayon', 'TUCP', 'ACT Teachers', '1PACMAN', 'TGP', 'DUMPER PTDA', 'Anakalusugan', 'Aksyon Dapat', 'BHW', 'Sulong Dignidad', 'Batang Quiapo', 'PBA', 'GILAS', 'Ako Ilokano Ako', 'Pamilyang Magsasaka', 'Click Party', 'Abante Bisdak', 'Manila Teachers', 'PAMANA', 'Nanay', 'KM Ngayon Na', 'Babae Ako', 'ARISE', 'Magdalo', 'APEC', 'MAGBUBUKID', 'SSS-GSIS Pensyonado', 'GABRIELA', 'Tingog', 'APAT-DAPAT', 'Ahon Mahirap', 'UGB', 'Akbayan', 'Agimat', 'PHILRECA', 'Kapuso PM', 'Ilocano Defenders', '1-Rider Party-list', 'TICTOK', 'Bayan Muna', 'Ang Probinsyano', 'BANAT', 'SBP', 'Buhay', 'Tulungan Tayo', 'SAGIP', 'BTS Bayaning Tsuper', 'Vendors', 'ACT-CIS', 'Aktibong Kaagapay', 'Asenso Pinoy', 'Solo Parents', 'Ang Komadrona', 'PROMDI', 'Pusong Pinoy', 'Kusug Tausug', 'Damayang Filipino', 'MPBL', 'ANGAT', 'Kalinga', 'Boses Party-list', 'Arangkada Pilipino', 'Aangat Tayo', 'OFW', 'BIDA KATAGUMPAY', 'KAMANGGAGAWA', 'BFF', 'Bunyog', 'AGRI', 'Senior Citizens', '4K', 'PBP', 'One Coop', 'CIBAC', 'BH - Bagong Henerasyon', '1AGILA', 'EDUAKSYON', 'Ang Tinig ng Seniors', 'BG Party-list', 'Pinoy Ako', 'H.E.L.P. PILIPINAS', 'Health Workers', "People's Champ", 'AA-Kasosyo Party', 'Solid North Party', 'ABAMIN', 'TRABAHO', 'ANGKASangga', 'TODA Aksyon', 'Turismo', 'Abono', 'ASAP NA', 'LINGAP', 'United Frontliners', 'Kasambahay', 'Tutok To WIn', 'Ako OFW', 'AGAP', '1TAHANAN', 'Coop-NATCCO', 'KABAYAN', '1Munti', 'PINOY WORKERS', 'API Party', 'Ako Bisaya', 'KAMALAYAN', 'Ako Tanod', 'Probinsyano Ako', 'KABABAIHAN', 'RAM', 'ALONA', 'Ako Bikol', 'GP (Galing sa Puso)', 'KAUNLAD PINOY', 'ABP', 'CWS', 'LPGMA', 'A TEACHER', 'SWERTE', 'Gabay', 'Malasakit@Bayanihan', 'Akay ni Sol', 'LUNAS', 'DIWA', 'PINUNO', 'Pamilya Muna', 'Bagong Pilipinas', 'Hugpong Federal', 'Tupad', 'Lang Kawal', 'Pamilya Ko', 'BBM', 'Heal PH', 'Abang Lingkod', 'MAGSASAKA', 'Maharlika', 'Uswag Ilonggo']

candidates_lowered = [candidate.lower() for candidate in Candidates]
partylists_lowered = [partylist.lower() for partylist in Partylists]

In [8]:
from enum import Enum
from typing import Union, Literal, Optional
# from datetime import datetime
from pydantic import BaseModel, Field, field_validator
# from rapidfuzz import process as fuzzy_process
import json

CANDIDATES = ",".join(candidates_lowered)
PARTYLISTS = ",".join(partylists_lowered)
CANDIDATE_SET = set(candidates_lowered)
PARTYLIST_SET = set(partylists_lowered)

class ToneType(str, Enum):
    ANGER = 'Anger'
    CONTEMPT = 'Contempt'
    DISGUST = 'Disgust'
    ENJOYMENT = 'Enjoyment'
    FEAR = 'Fear'
    SADNESS = 'Sadness'
    SURPRISE = 'Surprise'
    NEUTRAL = 'Neutral'

class PerceivedJudgement(str, Enum):
    INFAVOR = 'InFavor'
    UNSURE = 'Unsure'
    OPPOSEDTO = 'Opposedto'

class InfoItem(BaseModel):
    mentioned_candidate_AI: str | None = Field(description=f"A mentioned senatorial candidate or partylist within the content, only senatorial candidates of the 2025 midterm elections, no other years, no governors, mayors, or presidents, when a member of a partylist is mentioned, only state the partylist instead, dont state any other person other than those in the following exactly in the way it was written, no nicknames, no abbreviations, only must be in {CANDIDATES} or a partylist that only must be in {PARTYLISTS}")
    associated_word_AI: str | None = Field(description="A word associated with the candidate or partylist limited to at most one word.")
    tone_AI: ToneType = Field(description="Emotional tone or context of the text.")
    perceived_judgement_AI: PerceivedJudgement = Field(description="Perceived judgement regarding a candidate.")
    polarity_AI: float = Field(description="Polarity of the tone of the text, from -1 to 1.")
    is_spam: bool = Field(description="A boolean whether the content is spam, or has no relation to the philippine elections")

    @field_validator("polarity_AI")
    @classmethod
    def validate_polarity(cls, v):
        if v < -1 or v > 1:
            raise ValueError("Polarity needs to be between -1 and 1")
        return v

    def __str__(self) -> str:
        return json.dumps(self.model_dump(), default=str, indent=4)
    
    @field_validator("mentioned_candidate_AI")
    @classmethod
    def validate_mentioned_candidate(cls, v):
        if v is None or v == "":
            return v

        v_lowered = v.lower()
        
        if v_lowered not in CANDIDATE_SET and v_lowered not in PARTYLIST_SET or v_lowered == "none":
            raise ValueError("Candidate/Partylist needs to be in the provided list or None")
        return v_lowered

class ExtractInfo(BaseModel):
    candidates: list[InfoItem] = Field(description="A list of candidates found in the text with the corresponding tone evoked, judgement, and polarity towards the candidate")

print(CANDIDATES)
print(PARTYLISTS)
print(CANDIDATE_SET)
print(PARTYLIST_SET)

benhur abalos,jerome adonis,wilson amad,jocelyn andamo,bam aquino,ronnel arambulo,ernesto arellano,roberto ballon,abigail binay,jimmy bondoc,bong revilla,bonifacio bosita,arlene brosas,roy cabonegro,allen capuyan,teodoro casiño,france castro,pia cayetano,david d'angelo,angelo de alban,leody de guzman,ronald dela rosa,mimi doringo,arnel escobal,luke espiritu,mody floranda,marc gamboa,bong go,norberto gonzales,jesus hinlo jr.,gregorio honasan,relly jose jr.,panfilo lacson,raul lambino,lito lapid,wilbert t. lee,amirah lidasan,rodante marcoleta,imee marcos,norman marquez,eric martinez,richard mata,sonny matula,liza maza,heidi mendoza,jose montemayor jr.,subair mustapha,jose olivar,willie ong,manny pacquiao,kiko pangilinan,ariel querubin,apollo quiboloy,danilo ramos,willie revillame,vic rodriguez,nur-ana sahidulla,phillip salvador,tito sotto,michael tapado,francis tolentino,ben tulfo,erwin tulfo,mar valbuena,leandro verceles jr.,camille villar
makabayan,4ps,ppp,fpj panday bayanihan,kabataan

In [9]:
import instructor

random_text = "Hi po"
response = client.chat.completions.create(
    model=MODEL_ID,
    response_model=ExtractInfo,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Extract information from this response: {random_text}"},
    ],
    max_retries=3,
    temperature=0
)

print(f"Input Response:\n {random_text}\n\n")
print(f"Extracted Features:\n\n {response}")

Input Response:
 Hi po


Extracted Features:

 candidates=[InfoItem(mentioned_candidate_AI=None, associated_word_AI=None, tone_AI=<ToneType.NEUTRAL: 'Neutral'>, perceived_judgement_AI=<PerceivedJudgement.UNSURE: 'Unsure'>, polarity_AI=0.0, is_spam=True)]


In [10]:
for candidate in response.candidates:
    print(candidate.mentioned_candidate_AI)

None


# Setup the Dataset

## Columns to be Analyzed

In [18]:
# WARNING, THIS REINITIALIZES THE TWO VARIABLES BELOW
# WHEN THE CODE BELOW SAYS AN ERROR THAT SENTIMENT_MEMO AND SENTIMENT_DICT UNDEFINED, UNCOMMENT THE CODE HERE AND RUN THIS THEN RUN EVERYTHING BELOW THIS AGAIN

# sentiment_dict = {
#     "tweet_id" : [],
#     "mentioned_candidate_AI" : [],
#     "associated_word_AI" : [],
#     "tone_AI" : [],
#     "perceived_judgement_AI" : [],
#     "polarity_AI" : [],
#     "is_spam" : [],
# }

# sentiment_memo = set()

In [21]:
# Make function to get the sentiment polarities per answer

def get_sentiment(text: str) -> tuple[str, float] | None:
    response = client.chat.completions.create(
        model=MODEL_ID,
        response_model=ExtractInfo,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": f"Extract: {text}"
            },
        ],
        temperature=0,
        max_retries=3
    )

    return response

# Get sentiment for all tweets in Dataset

In [24]:
"""
is_sentimented is a set containing all tweetIds that were sentimented before.
modifies tweet_sentiments in place.
"""
def get_sentiments_df(tweets_df: pd.DataFrame, tweet_sentiments: pd.DataFrame,  is_sentimented: set[str], is_save_csv: bool = False):
    for tweet_id in tweets_df['Tweet ID']:
        if tweet_id not in is_sentimented:
            print(f'Analysing {tweet_id}')
            tweet_row = tweets_df.loc[tweets_df['Tweet ID']  == tweet_id]
            tweet_content = tweet_row['Content'].values[0]
            print(f'Content {tweet_content}')
            sentiment = get_sentiment(tweet_content)
            for sentiment in sentiment.candidates:
                new_row = {key : value for key, value in sentiment}
                new_row['tweet_id'] = tweet_id
                print(new_row)
                new_row_df = pd.DataFrame(new_row, index=[0])
                tweet_sentiments = pd.concat([tweet_sentiments, new_row_df], ignore_index=True)
                for key, value in sentiment:
                    print(key, value)
                tweet_sentiments_df = tweet_sentiments # hack to update df
            is_sentimented.add(tweet_id)
            if is_save_csv:
                tweet_sentiments.to_csv("tweet_sentiments.csv", index=False)
        else:
            print(f'{tweet_id} already sentimented')

In [27]:
tweets_df = pd.read_csv("X_Tweets_Filtered.csv")

In [30]:
# update memo with all tweet_ids already done in tweet_sentiments
tweet_sentiments_df = pd.read_csv("tweet_sentiments.csv")
sentiments_in_csv = set(tweet_sentiments_df['tweet_id'])

if sentiments_in_csv != sentiment_memo:
    sentiment_memo = sentiments_in_csv
    print("updated memo")
else:
    print("sentiment memo still has latest tweet_ids")

updated memo


In [33]:
get_sentiments_df(tweets_df, tweet_sentiments_df, sentiment_memo, is_save_csv=True) 


Sus ginoo Pilipinas…#Halalan2025
{'mentioned_candidate_AI': None, 'associated_word_AI': 'scammer', 'tone_AI': <ToneType.ANGER: 'Anger'>, 'perceived_judgement_AI': <PerceivedJudgement.OPPOSEDTO: 'Opposedto'>, 'polarity_AI': -0.8, 'is_spam': False, 'tweet_id': 'tweet_id:1874479972167082263'}
mentioned_candidate_AI None
associated_word_AI scammer
tone_AI ToneType.ANGER
perceived_judgement_AI PerceivedJudgement.OPPOSEDTO
polarity_AI -0.8
is_spam False
{'mentioned_candidate_AI': None, 'associated_word_AI': 'kawatan', 'tone_AI': <ToneType.ANGER: 'Anger'>, 'perceived_judgement_AI': <PerceivedJudgement.OPPOSEDTO: 'Opposedto'>, 'polarity_AI': -0.8, 'is_spam': False, 'tweet_id': 'tweet_id:1874479972167082263'}
mentioned_candidate_AI None
associated_word_AI kawatan
tone_AI ToneType.ANGER
perceived_judgement_AI PerceivedJudgement.OPPOSEDTO
polarity_AI -0.8
is_spam False
Analysing tweet_id:1873763656162480360
Content Last day of the year. Iwan na natin sina Pia Cayetano, Bato Dela Rosa, Bong Go, L

InstructorRetryException: <failed_attempts>

<generation number="1">
<exception>
    1 validation error for ExtractInfo
candidates.4.mentioned_candidate_AI
  Value error, Candidate/Partylist needs to be in the provided list or None [type=value_error, input_value='Bato', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error
</exception>
<completion>
    ChatCompletion(id='chatcmpl-c798c61f-27ce-4de3-9ff0-1508563f13b0', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n"candidates": [\n    {\n      "mentioned_candidate_AI": "Kiko Pangilinan",\n      "associated_word_AI": "handang",\n      "tone_AI": "Enjoyment",\n      "perceived_judgement_AI": "InFavor",\n      "polarity_AI": 0.8,\n      "is_spam": false\n    },\n    {\n      "mentioned_candidate_AI": "Bam Aquino",\n      "associated_word_AI": "handang",\n      "tone_AI": "Enjoyment",\n      "perceived_judgement_AI": "InFavor",\n      "polar

# Turn into Dataframes and save to CSVs

In [135]:
halalan2025_batch1 = pd.DataFrame(sentiment_dict)
halalan2025_batch1.to_csv('finalbatch_hashed_Halalan_Sentiments_LLMA4MVRICK.csv')
halalan2025_batch1

,tweet_id,mentioned_candidate_AI,associated_word_AI,tone_AI,perceived_judgement_AI,polarity_AI,is_spam
0,tweet_id:1910902241202184588,kabayan,Vote,ToneType.NEUTRAL,PerceivedJudgement.UNSURE,0.0,True
1,tweet_id:1919334684942639363,kiko pangilinan,WIN,ToneType.ENJOYMENT,PerceivedJudgement.INFAVOR,0.9,False


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=379262c2-f7d6-45ba-811e-59f3f72f9143' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>